In [1]:
import torch

from torch_openreml import MarginalREML
from torch_openreml.covariance import DummyMatrix, ScalarMatrix, CovariancePropagation, Sum

n, p = 50, 2

y = torch.randn(n)
X = torch.randn(n, p)

Z = DummyMatrix(["a", "b"] * 25)

V = Sum(
    CovariancePropagation(
        Z,
        ScalarMatrix(2),
    ),
    ScalarMatrix(n),
)

reml = MarginalREML(V)

theta_start = torch.zeros(V.num_free_params)

theta_hat, beta_hat, n_iter = reml.optimize(
    y,
    X,
    theta_start,
    verbose=2,
)

 ⏱ 00:00 | ⚡ ?it/s

Iter 1:  ⏱ 00:00 | ⚡ ?it/s

Iter 1:  ⏱ 00:00 | ⚡ 223.52it/s

Iter 1:  ⏱ 00:00 | ⚡ 183.35it/s

Iter 2:  ⏱ 00:00 | ⚡ 118.88it/s

...

Iter 9:  ⏱ 00:00 | ⚡ 317.21it/s

Iter 9:  ⏱ 00:00 | ⚡ 308.38it/s

Iter 9:  ⏱ 00:00 | ⚡ 302.46it/s

Iter 9:  ⏱ 00:00 | ⚡ 300.19it/s


∥∇∥:       4.8875, ∥Δ∥: 2.4504, η: 1.00, ∥Δᶜ∥: 2.4504, log 𝓛: -32.6976
∥∇∥:       4.2734, ∥Δ∥: 2.5578, η: 1.00, ∥Δᶜ∥: 2.5578, log 𝓛: -32.7835 (-0.0859)
∥∇∥:       1.6544, ∥Δ∥: 3.1305, η: 1.00, ∥Δᶜ∥: 3.1305, log 𝓛: -32.7630 (+0.0204)
∥∇∥:       5.0415, ∥Δ∥: 7.6494, η: 1.00, ∥Δᶜ∥: 7.6494, log 𝓛: -33.0111 (-0.2481)
∥∇∥:       2.0004, ∥Δ∥: 28507.4688, η: 1.00, ∥Δᶜ∥: 28507.4688, log 𝓛: -41.6217 (-8.6106)
∥∇∥:       5.6378, ∥Δ∥: 0.0526, η: 1.00, ∥Δᶜ∥: 0.0526, log 𝓛: -33.1422 (+8.4795)
∥∇∥:       0.2862, ∥Δ∥: 0.0030, η: 1.00, ∥Δᶜ∥: 0.0030, log 𝓛: -32.9890 (+0.1532)
∥∇∥:       0.0008, ∥Δ∥: 0.0000, η: 1.00, ∥Δᶜ∥: 0.0000, log 𝓛: -32.9885 (+0.0004)
∥∇∥:       0.0000, ∥Δ∥: 0.0000, η: 1.00, ∥Δᶜ∥: 0.0000, log 𝓛: -32.9885 (+0.0000)

[∇: score, Δ: 𝐉⁻¹∇, η: learning rate, Δᶜ: clip(𝛉 + ηΔ, lb, ub) - 𝛉, 𝓛: restricted likelihood]

✓ Converged at iteration 9


In [2]:
theta_last = reml.get_theta(select="last")
theta_best = reml.get_theta(select="best")

beta_last = reml.get_beta(select="last")
beta_best = reml.get_beta(select="best")

In [3]:
beta_hat = reml.blue(y, X, theta_hat)

In [4]:
y_hat = reml.predict(
    y,
    X,
    theta_hat,
)

In [5]:
e = reml.residual(
    y,
    X,
    theta_hat,
)

In [6]:
loglik = reml.loglik(y, X, theta_hat)

In [7]:
import torch

from torch_openreml import MarginalREML
from torch_openreml.utils import augment, n_distinct

from torch_openreml.covariance import (
    DummyMatrix,
    IdentityMatrix,
    ScalarMatrix,
    Sum,
    CovariancePropagation,
    KroneckerProduct,
)

from torch_openreml.example_data import john_alpha

# --- response ---
y = torch.tensor(john_alpha["yield"].values)

# --- fixed effects ---
X = augment(
    torch.ones(len(john_alpha), 1),
    DummyMatrix(john_alpha["rep"], drop_first=True)()
)

# --- random effect design matrices ---
Z_gen = DummyMatrix(john_alpha["gen"])
Z_rep_block = DummyMatrix(john_alpha["rep"], john_alpha["block"])

# --- covariance components ---
G_gen = ScalarMatrix(n_distinct(john_alpha["gen"]))
G_rep = IdentityMatrix(n_distinct(john_alpha["rep"]))
G_block = ScalarMatrix(n_distinct(john_alpha["block"]))

R = ScalarMatrix(len(john_alpha))

# --- marginal covariance ---
V = Sum(
    CovariancePropagation(Z_gen, G_gen),
    CovariancePropagation(
        Z_rep_block,
        KroneckerProduct(G_rep, G_block)
    ),
    R
)

# --- REML fit ---
reml = MarginalREML(V)

theta_start = torch.zeros(V.num_free_params)

theta_hat, beta_hat, n_iter = reml.optimize(
    y,
    X,
    theta_start,
    verbose=2,
)

# --- results ---
print("theta:", theta_hat)

print("variance components:", V.build_params(theta_hat))

print("fixed effects:", beta_hat)

print("loglik:", reml.loglik(y, X, theta_hat))

 ⏱ 00:00 | ⚡ ?it/s

Iter 1:  ⏱ 00:00 | ⚡ ?it/s

Iter 1:  ⏱ 00:00 | ⚡ 445.44it/s

Iter 1:  ⏱ 00:00 | ⚡ 340.58it/s

Iter 2:  ⏱ 00:00 | ⚡ 172.78it/s

...

Iter 19:  ⏱ 00:00 | ⚡ 358.45it/s

Iter 19:  ⏱ 00:00 | ⚡ 353.52it/s

Iter 19:  ⏱ 00:00 | ⚡ 350.22it/s

Iter 19:  ⏱ 00:00 | ⚡ 349.10it/s


∥∇∥:      41.8197, ∥Δ∥: 8.9438, η: 1.00, ∥Δᶜ∥: 8.9438, log 𝓛: -34.3129
∥∇∥:  315227.4375, ∥Δ∥: 0.8838, η: 1.00, ∥Δᶜ∥: 0.8838, log 𝓛: -201185.3906 (-201151.0777)
∥∇∥:  119230.2656, ∥Δ∥: 0.8519, η: 1.00, ∥Δᶜ∥: 0.8519, log 𝓛: -71844.5469 (+129340.8438)
∥∇∥:   44273.9336, ∥Δ∥: 0.8143, η: 1.00, ∥Δᶜ∥: 0.8143, log 𝓛: -26065.6270 (+45778.9199)
∥∇∥:   16331.3926, ∥Δ∥: 0.7587, η: 1.00, ∥Δᶜ∥: 0.7587, log 𝓛: -9457.4141 (+16608.2129)
∥∇∥:    6001.7085, ∥Δ∥: 0.7142, η: 1.00, ∥Δᶜ∥: 0.7142, log 𝓛: -3393.0168 (+6064.3972)
∥∇∥:    2194.9878, ∥Δ∥: 0.7031, η: 1.00, ∥Δᶜ∥: 0.7031, log 𝓛: -1181.3339 (+2211.6830)
∥∇∥:     793.8170, ∥Δ∥: 0.6952, η: 1.00, ∥Δᶜ∥: 0.6952, log 𝓛: -382.5103 (+798.8235)
∥∇∥:     278.6989, ∥Δ∥: 0.6715, η: 1.00, ∥Δᶜ∥: 0.6715, log 𝓛: -102.3058 (+280.2045)
∥∇∥:      90.4160, ∥Δ∥: 0.5957, η: 1.00, ∥Δᶜ∥: 0.5957, log 𝓛: -11.4295 (+90.8763)
∥∇∥:      23.9232, ∥Δ∥: 0.4006, η: 1.00, ∥Δᶜ∥: 0.4006, log 𝓛:  12.6788 (+24.1083)
∥∇∥:       3.8678, ∥Δ∥: 0.1392, η: 1.00, ∥Δᶜ∥: 0.1392, log 𝓛:  16.5908

In [8]:
scores = [
    torch.norm(s).item()
    for s in reml.history["score"]
]

logliks = [
    ll.item()
    for ll in reml.history["loglik"]
]

print(
    "Score norms:",
    [f"{s:.6f}" for s in scores],
)

print(
    "Log-likelihoods:",
    [f"{ll:.4f}" for ll in logliks],
)

Score norms: ['41.819736', '315227.437500', '119230.265625', '44273.933594', '16331.392578', '6001.708496', '2194.987793', '793.817017', '278.698944', '90.416023', '23.923166', '3.867793', '0.244776', '0.010351', '0.000450', '0.000027', '0.000051', '0.000069', '0.000054']
Log-likelihoods: ['-34.3129', '-201185.3906', '-71844.5469', '-26065.6270', '-9457.4141', '-3393.0168', '-1181.3339', '-382.5103', '-102.3058', '-11.4295', '12.6788', '16.5908', '16.8077', '16.8099', '16.8097', '16.8101', '16.8098', '16.8101', '16.8101']
